# B. 기술조사 에이전트

| | |
|---|---|
| **담당** | RAG(가이드 표 O) + TRL 7~9 판정용 보조 Web Search |
| **선행 노드** | A |
| **출력 State 키** | `tech_research`, `tech_references` |

TurboQuant·InfiniGen 원문을 하이브리드 검색(BM25+FAISS)으로 찾아 개요·범위·한계를 "추출"하고, TRL을 "판정"한다.
TRL 1~6은 원문 검색만으로 판정 가능하지만, 7~9(공식 배포·상용 지원)는 논문 발표 **이후**에 일어나는 일이라 원문에 없다 — 좁은 웹 검색 2건으로만 보조 확인한다.

design doc 참고: 3-2절 B, 2-1절(TRL), 3-1절("B는 RAG가 핵심, TRL 7~9 판정만 좁은 웹 검색으로 보조")

이 노트북 끝에서 만든 함수는 `src/nodes_b.py`로 저장되고, `05_full_graph_run.ipynb`가 그걸 가져다 쓴다.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import config, prompts
from src.ingest import build_tech_retriever, format_docs_for_prompt
from src.node_utils import run_web_search
from src.schemas import TechResearch

## 1. RAG 리트리버 준비

`data/papers/turboquant.pdf`, `data/papers/infinigen.pdf`가 있어야 한다.
없으면 터미널에서 `bash ../scripts/download_papers.sh`부터 실행할 것.

Qwen3-Embedding-0.6B를 처음 부르면 모델을 다운로드하느라 몇 분 걸릴 수 있다.

In [ ]:
tech_retriever = build_tech_retriever()
print("리트리버 준비 완료")

## 2. 프롬프트 확인

`prompts.py`에 이미 정의돼 있다. 2-1절 TRL 판정 규칙이 그대로 지시문으로 들어가 있는지 확인만 한다.

In [ ]:
print(prompts.TECH_RESEARCH_PROMPT)

## 3. 노드 함수 정의

시그니처는 항상 `(state: GraphState) -> dict` — 반환한 dict만 State에 병합된다.
`llm`·`tech_retriever`·`web_search_tool`은 클로저로 주입해서, 나중에 테스트할 때 가짜 객체를 넣을 수 있게 한다.

In [ ]:
def make_node_b(llm, tech_retriever, web_search_tool):
    """B. 기술조사 노드를 만든다. llm/tech_retriever/web_search_tool을
    미리 받아 클로저로 갖고 있다가, 실제 노드 함수(node_b_tech_research)가
    호출될 때 사용한다."""
    structured_llm = llm.with_structured_output(TechResearch)

    def node_b_tech_research(state):
        queries = [
            "TurboQuant KV cache quantization bits distortion",
            "InfiniGen KV cache offloading prefetch speedup",
        ]
        all_docs = []
        for q in queries:
           docs = tech_retriever.invoke(q)
           all_docs.extend(docs[:5])   # 검색어 하나당 최대 5개로 미리 제한
        context = format_docs_for_prompt(all_docs, max_docs=10)

        deployment_search_results = run_web_search(
            web_search_tool,
            [
                "TurboQuant official runtime production deployment",
                "InfiniGen official runtime vLLM llama.cpp support",
            ],
        )

        prompt = prompts.TECH_RESEARCH_PROMPT.format(
            context=context, deployment_search_results=deployment_search_results
        )
        prompt += (
           "\n\n[추가 유의사항] 'offloading' 방식이라도 실험 환경이 GPU 서버 + "
           "호스트 CPU 메모리라면 이는 데이터센터 환경이지 온디바이스가 아니다. "
           "온디바이스로 인정하려면 스마트폰·임베디드 보드·통합 메모리 노트북에서의 "
           "실측/배포 근거가 있어야 한다."
       )
        result = structured_llm.invoke(prompt)

        refs = [
            {"source": d.metadata.get("source_name", "unknown"), "detail": d.page_content[:200]}
            for d in all_docs
        ]
        refs.append({"source": "web_search(TRL 배포 확인용)",
                      "detail": deployment_search_results[:200]})
        return {
            "tech_research": {
                "TurboQuant": result.turboquant.model_dump(),
                "InfiniGen": result.infinigen.model_dump(),
            },
            "tech_references": refs,
        }

    return node_b_tech_research

## 4. 배선 테스트 — API 키 없이

가짜 LLM을 넣어서 함수가 에러 없이 도는지, 반환 dict 모양이 맞는지만 확인한다.
실제 판정 품질은 여기서 알 수 없다 — 그건 5번의 "실제 LLM 테스트"에서 본다.

In [ ]:
from src.schemas import ResearchResult, TRLAssessment

class FakeStructuredLLM:
    def __init__(self, output):
        self.output = output
    def invoke(self, prompt):
        return self.output

class FakeLLM:
    def with_structured_output(self, schema_cls):
        rr = ResearchResult(
            overview="가짜 개요", scope="가짜 범위", limitations="가짜 한계",
            trl_assessment=TRLAssessment(trl_ondevice=4, trl_global=6, evidence_status="found"),
        )
        return FakeStructuredLLM(TechResearch(turboquant=rr, infinigen=rr))

class FakeRetriever:
    def invoke(self, query):
        from langchain_core.documents import Document
        return [Document(page_content=f"가짜 문서 for '{query}'", metadata={"source_name": "fake.pdf"})]

class FakeWebSearchTool:
    def invoke(self, args):
        return f"가짜 검색 결과: {args['query']}"

node_b = make_node_b(FakeLLM(), FakeRetriever(), FakeWebSearchTool())
result = node_b({})
assert "TurboQuant" in result["tech_research"]
assert len(result["tech_references"]) == 3  # RAG 2쿼리 + 배포확인 웹서치 1건
print("배선 OK")
print(result["tech_research"]["TurboQuant"])

## 5. 실제 LLM 테스트

`.env`에 `OPENAI_API_KEY`, `TAVILY_API_KEY`가 있어야 실행된다. 없으면 건너뛴다.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("OPENAI_API_KEY") and os.environ.get("TAVILY_API_KEY"):
    from langchain.chat_models import init_chat_model
    from langchain_tavily import TavilySearch

    real_llm = init_chat_model(config.LLM_MODEL, model_provider=config.LLM_PROVIDER, temperature=0)
    real_web_search = TavilySearch(max_results=5)

    node_b_real = make_node_b(real_llm, tech_retriever, real_web_search)
    real_result = node_b_real({})
    print(real_result["tech_research"])
else:
    print("API 키 없음 - 이 셀은 건너뜀. .env를 채운 뒤 다시 실행할 것.")

## 6. 파일로 저장

이 노트북에서 정의한 `make_node_b`를 `src/nodes_b.py`로 저장한다.
`05_full_graph_run.ipynb`가 이 파일을 import해서 그래프에 조립한다.

**코드를 고쳤으면 이 셀을 다시 실행해서 파일을 갱신해야 한다** — 위에서 함수를 고치고 이 셀을 안 돌리면 05번 노트북은 옛날 버전을 계속 쓴다.

In [ ]:
import inspect

TARGET = "../src/nodes_b.py"

# 심볼 유실 감지. 아래 parts 목록은 하드코딩이라, 누가 src/nodes_b.py 을 직접 고쳐
# 함수·상수를 더해 놓으면 저장하는 순간 그게 조용히 사라진다(2026-09-22 실제 발생).
# 사라진 이름이 있으면 여기서 알린다 - 에러가 안 나서 안 보이는 게 진짜 위험이다.
def _symbols(path):
    import ast, os
    if not os.path.exists(path):
        return set()
    out = set()
    for n in ast.parse(open(path, encoding="utf-8").read()).body:
        if isinstance(n, ast.FunctionDef):
            out.add(n.name)
        elif isinstance(n, ast.Assign):
            out |= {t.id for t in n.targets if isinstance(t, ast.Name)}
    return out


def _src(obj):
    """getsource 결과의 꼬리 개행을 없앤다. Jupyter 는 셀 끝에 개행을 붙이고
    스크립트 실행은 안 붙여서, 그대로 쓰면 환경마다 결과 파일이 달라진다."""
    return inspect.getsource(obj).rstrip("\n")


_before = _symbols(TARGET)

q3 = chr(34) * 3
HEADER = (
    q3 + "B. 기술조사 노드 - 01_agent_B_tech_research.ipynb에서 생성됨.\n"
    "이 파일을 직접 고치지 말고, 노트북에서 고친 뒤 저장 셀을 다시 실행할 것." + q3 + "\n\n"
    "from src import prompts\n"
    "from src.ingest import format_docs_for_prompt\n"
    "from src.node_utils import run_web_search\n"
    "from src.schemas import TechResearch\n\n\n"
)

parts = [
    _src(make_node_b),
]

with open(TARGET, "w", encoding="utf-8") as f:
    f.write(HEADER + "\n\n\n".join(parts) + "\n")   # 정의 사이는 빈 줄 2개(PEP8)

_lost = sorted(_before - _symbols(TARGET))
if _lost:
    print(f"🔴 이번 저장으로 {TARGET} 에서 사라진 심볼: {_lost}")
    print("   노트북이 .py 보다 낡았다는 뜻이다. git diff 로 확인하고,")
    print("   의도한 삭제가 아니면 git checkout 으로 되돌린 뒤 노트북부터 맞출 것.")
else:
    print(f"{TARGET} 저장 완료 (심볼 유실 없음)")